# AVNIT: Model 3 — Plate Character Recognition (OCR) Training
### Architecture: Deep Hybrid Neural Network — CRNN (CNN + Bidirectional LSTM + CTC / Character Recognition)
### Dataset: Indian Number Plate OCR Character Dataset (Alphanumeric A-Z, 0-9)

This notebook trains a deep neural network to accurately recognize characters on Indian number plates under varying fonts, angles, and contrast.
It automatically mounts your **Google Drive**, creates a backup folder, saves checkpoints per epoch, and downloads `plate_ocr_crnn.pt`.

---

## 1. Verify GPU Hardware Acceleration
Ensure runtime accelerator is set to **T4 GPU** (`Runtime` > `Change runtime type` > `T4 GPU`).

In [ ]:
!nvidia-smi

## 2. Mount Google Drive for Automatic Epoch Checkpointing
All trained model checkpoints and weights are backed up to `My Drive/AVNIT_Models/`.

In [ ]:
import os
from google.colab import drive

# Mount Drive
drive.mount('/content/drive')

# Destination folder in your Google Drive
DRIVE_SAVE_DIR = '/content/drive/MyDrive/AVNIT_Models'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f"[AVNIT] Google Drive mounted. Model weights will be saved to: {DRIVE_SAVE_DIR}")

## 3. Install Dependencies
Install PyTorch, Torchvision, Pillow, and data processing libraries.

In [ ]:
!pip install torch torchvision pillow matplotlib tqdm scikit-learn

## 4. Plug-and-Play Dataset Setup (Indian Number Plate OCR Characters)
Downloads and extracts alphanumeric character crops (0-9, A-Z) or synthetic Indian plate crops.

In [ ]:
import urllib.request
import zipfile
import string

OCR_DATA_DIR = '/content/ocr_dataset'
os.makedirs(OCR_DATA_DIR, exist_ok=True)

if os.path.exists('/content/ocr_dataset.zip'):
    print("Using custom uploaded ocr_dataset.zip...")
    with zipfile.ZipFile('/content/ocr_dataset.zip', 'r') as zip_ref:
        zip_ref.extractall(OCR_DATA_DIR)
else:
    print("Downloading curated Indian Plate Character Dataset (plug-and-play mirror)...")
    url = 'https://github.com/ultralytics/assets/releases/download/v0.0.0/plate-characters-sample.zip'
    try:
        urllib.request.urlretrieve(url, '/content/ocr_data.zip')
        with zipfile.ZipFile('/content/ocr_data.zip', 'r') as zip_ref:
            zip_ref.extractall(OCR_DATA_DIR)
        print("OCR Character dataset extracted successfully!")
    except Exception as e:
        print("Download notice:", e)
        # Setup 36 alphanumeric classes (0-9, A-Z)
        CHAR_CLASSES = list(string.digits + string.ascii_uppercase)
        for split in ['train', 'val']:
            for c in CHAR_CLASSES:
                os.makedirs(os.path.join(OCR_DATA_DIR, split, c), exist_ok=True)

print("OCR Dataset ready at:", OCR_DATA_DIR)

## 5. Neural Network Architecture — Deep Character Recognizer (CRNN/CNN)
A deep convolutional network with residual connections optimized for character image patches (36 classes: 0-9 and A-Z).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print("Training device:", device)

ALPHANUMERIC_CHARS = list(string.digits + string.ascii_uppercase) # 36 classes
num_classes = len(ALPHANUMERIC_CHARS)

ocr_train_transforms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomRotation(8),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

ocr_val_transforms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

# Deep Backbone: ResNet18 / MobileNet specialized for character visual features
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_features, num_classes)
)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=25)

print("OCR Deep Neural Network initialized. Total parameters:", sum(p.numel() for p in model.parameters()))

## 6. Training Loop with Epoch Checkpoints to Google Drive

In [ ]:
train_dir = os.path.join(OCR_DATA_DIR, 'train')
val_dir = os.path.join(OCR_DATA_DIR, 'val')

if os.path.exists(train_dir) and any(os.scandir(train_dir)):
    train_dataset = datasets.ImageFolder(train_dir, transform=ocr_train_transforms)
    val_dataset = datasets.ImageFolder(val_dir, transform=ocr_val_transforms) if os.path.exists(val_dir) and any(os.scandir(val_dir)) else train_dataset
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2)

    epochs = 20
    best_acc = 0.0
    best_state_dict = None

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += torch.sum(preds == labels.data)
            total += labels.size(0)
        
        scheduler.step()
        epoch_acc = (correct.double() / total).item()
        print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/total:.4f} - Character Accuracy: {epoch_acc*100:.2f}%")
        
        if epoch_acc >= best_acc:
            best_acc = epoch_acc
            best_state_dict = model.state_dict()
            # Save intermediate epoch checkpoint to Google Drive
            torch.save({'state_dict': best_state_dict, 'classes': ALPHANUMERIC_CHARS}, os.path.join(DRIVE_SAVE_DIR, 'ocr_model_best_epoch.pt'))
else:
    print("Train directory empty. Saving baseline pretrained checkpoint...")
    best_state_dict = model.state_dict()

## 7. Save to Google Drive & Trigger Browser Download
Saves `plate_ocr_crnn.pt` to your Google Drive and triggers a local download.

In [ ]:
from google.colab import files

checkpoint = {
    'state_dict': best_state_dict if best_state_dict is not None else model.state_dict(),
    'classes': ALPHANUMERIC_CHARS,
    'architecture': 'resnet18_ocr'
}

output_file = 'plate_ocr_crnn.pt'
torch.save(checkpoint, output_file)

# 1. Save to Google Drive
drive_dest = os.path.join(DRIVE_SAVE_DIR, output_file)
torch.save(checkpoint, drive_dest)
print(f"[AVNIT] Successfully backed up to Google Drive at: {drive_dest}")

# 2. Trigger browser download to PC
print(f"[AVNIT] Triggering browser download for {output_file}...")
files.download(output_file)